# BRCA1 preneoplastic analysis + PBS-2 noise cells

Sibling to `brca1-preneoplastic-null.ipynb`. Re-runs the **same Seurat CCA + edgeR pipeline** (Fig 4A–C / Appendix S1), but expands each sample to:

> **QC-pass cells** (`is_noise == 0`) **+ PBS-2 noise cells** (dormant: low mito %, low total RNA, moderate gene count).

PBS-2 labels use `classify_noise_subtypes` on the noise-only population (project defaults).

| vs null notebook | This experiment |
|---|---|
| Cells | paper QC only | QC-pass + PBS-2 |
| R cell QC | SampleStats thresholds | **skipped** (barcodes pre-filtered in Python) |
| Integration / clustering / DE | identical `NormBRCA1_faithful.R` | same script, `skip_cell_qc=TRUE` |
| Output dir | `.../brca1_preneoplastic_seurat/` | `.../brca1_preneoplastic_seurat_pbs2/` |

**Note:** Author epithelial cluster IDs (2, 4, 5, 8) may not map 1:1 after adding PBS-2 cells; compare figures and cluster composition to the null run rather than assuming identical cluster numbering.

**Prereqs:** same as the null notebook (GSE161529 raw + features + annotations, R/Seurat/edgeR, >=64 GB RAM).

## 0. Initializations

In [ ]:
from __future__ import annotations

import gzip
import shutil
import subprocess
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy.io import mmread, mmwrite

from signals_in_the_noise.analysis.noise_phenotypes import classify_noise_subtypes
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.preprocessing.gse161529 import GSE161529

HERE = Path.cwd()
CANDIDATES = [
    HERE,
    HERE / "notebooks" / "GSE161529" / "necb",
    Path(r"c:\Users\silly\GitHub\signals-in-the-noise\notebooks\GSE161529\necb"),
    Path(r"D:\GitHub\signals-in-the-noise\notebooks\GSE161529\necb"),
]
R_SCRIPT = next(
    (p / "NormBRCA1_faithful.R" for p in CANDIDATES if (p / "NormBRCA1_faithful.R").exists()),
    None,
)
if R_SCRIPT is None:
    raise FileNotFoundError("Could not locate NormBRCA1_faithful.R next to this notebook.")

RAW_DIR = get_data_path("raw") / "GSE161529_RAW"
FEATURES = get_data_path("raw") / "GSE161529_features.tsv.gz"
ANNOTATIONS = get_data_path("GSE161529_annotations_df.csv")

WORK_DIR = get_data_path("GSE161529_analysis_cache") / "brca1_preneoplastic_seurat_pbs2"
DATA_ROOT = WORK_DIR / "Data"
OUT_DIR = WORK_DIR / "output"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("R_SCRIPT :", R_SCRIPT)
print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR  :", OUT_DIR)

In [ ]:
# Author sample IDs -> (AnnData cache filename, raw GEO stem)
SAMPLE_MAP = {
    "N-0019-total": ("GSM4909254_N-PM0019-Total.h5ad", "GSM4909254_N-PM0019-Total"),
    "N-0233-total": ("GSM4909265_N-PM0233-Total.h5ad", "GSM4909265_N-PM0233-Total"),
    "N-0092-total": ("GSM4909253_N-PM0092-Total.h5ad", "GSM4909253_N-PM0092-Total"),
    "N-0230.17-total": ("GSM4909261_N-PM0230-Total.h5ad", "GSM4909261_N-PM0230-Total"),
    "N-0093-total": ("GSM4909257_N-PM0095-Total.h5ad", "GSM4909257_N-PM0095-Total"),
    "N-0123-total": ("GSM4909268_N-MH0023-Total.h5ad", "GSM4909268_N-MH0023-Total"),
    "N-0064-total": ("GSM4909263_N-MH0064-Total.h5ad", "GSM4909263_N-MH0064-Total"),
    "N-0169-total": ("GSM4909266_N-MH0169-Total.h5ad", "GSM4909266_N-MH0169-Total"),
    "B1-0894": ("GSM4909277_B1-KCF0894.h5ad", "GSM4909277_B1-KCF0894"),
    "B1-0033": ("GSM4909278_B1-MH0033.h5ad", "GSM4909278_B1-MH0033"),
    "B1-0023": ("GSM4909279_B1-MH0023.h5ad", "GSM4909279_B1-MH0023"),
    "B1-0090": ("GSM4909280_B1-MH0090.h5ad", "GSM4909280_B1-MH0090"),
}

assert ANNOTATIONS.exists(), f"Missing annotations: {ANNOTATIONS}"
assert FEATURES.exists(), f"Missing features: {FEATURES}"
assert RAW_DIR.exists(), f"Missing RAW dir: {RAW_DIR}"

gse = GSE161529()

## 1. Select QC-pass + PBS-2 barcodes and write 10X folders

For each of the 12 samples:
1. Classify noise subtypes on `is_noise == 1` cells.
2. Keep cells with `is_noise == 0` **or** `pbs-2 == True`.
3. Subset the raw GEO matrix to those barcodes (shared features file unchanged).
4. R is run with `skip_cell_qc=TRUE` so SampleStats thresholds are not re-applied.

In [ ]:
def link_or_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        dst.symlink_to(src.resolve())
    except OSError:
        shutil.copy2(src, dst)


def select_real_plus_pbs2(adata: ad.AnnData) -> ad.AnnData:
    """Return QC-pass cells plus PBS-2 noise cells; annotate cell_class."""
    adata = adata.copy()
    adata.obs["pbs-2"] = False
    noise = adata[adata.obs["is_noise"] == 1]
    if noise.n_obs > 0:
        noise = classify_noise_subtypes(noise.copy())
        adata.obs.loc[noise.obs_names, "pbs-2"] = noise.obs["pbs-2"].to_numpy()

    keep = (adata.obs["is_noise"] == 0) | adata.obs["pbs-2"].astype(bool)
    out = adata[keep].copy()
    out.obs["cell_class"] = np.where(out.obs["is_noise"] == 0, "qc_pass", "pbs-2")
    return out


def write_filtered_10x(stem: str, keep_barcodes: list[str], out_dir: Path) -> None:
    """Subset the raw GEO matrix to keep_barcodes; reuse shared features."""
    out_dir.mkdir(parents=True, exist_ok=True)
    barcodes_path = RAW_DIR / f"{stem}-barcodes.tsv.gz"
    matrix_path = RAW_DIR / f"{stem}-matrix.mtx.gz"
    if not barcodes_path.exists() or not matrix_path.exists():
        raise FileNotFoundError(f"Missing 10X files for {stem}")

    with gzip.open(barcodes_path, "rt") as f:
        all_barcodes = [line.strip() for line in f if line.strip()]

    keep_set = set(keep_barcodes)
    keep_idx = [i for i, bc in enumerate(all_barcodes) if bc in keep_set]
    if len(keep_idx) != len(keep_set):
        missing = keep_set - set(all_barcodes)
        raise RuntimeError(
            f"{stem}: {len(missing)} selected barcodes not found in raw 10X "
            f"(e.g. {list(missing)[:3]})"
        )

    # Matrix Market is genes x cells
    mat = mmread(matrix_path).tocsc()
    mat = mat[:, keep_idx]

    with gzip.open(out_dir / "barcodes.tsv.gz", "wt") as f:
        for i in keep_idx:
            f.write(all_barcodes[i] + "\n")

    mtx_tmp = out_dir / "matrix.mtx"
    mmwrite(mtx_tmp, mat)
    with open(mtx_tmp, "rb") as fin, gzip.open(str(mtx_tmp) + ".gz", "wb") as fout:
        shutil.copyfileobj(fin, fout)
    mtx_tmp.unlink()

    link_or_copy(FEATURES, out_dir / "features.tsv.gz")

In [ ]:
summary_rows = []

for sample_id, (h5ad_name, stem) in SAMPLE_MAP.items():
    print(f"Preparing {sample_id} ({stem})...")
    full = gse.get_dataset(h5ad_name)
    if full.n_obs == 0:
        raise RuntimeError(f"Empty AnnData for {h5ad_name}")

    selected = select_real_plus_pbs2(full)
    n_qc = int((selected.obs["cell_class"] == "qc_pass").sum())
    n_pbs2 = int((selected.obs["cell_class"] == "pbs-2").sum())
    n_noise = int((full.obs["is_noise"] == 1).sum())

    sample_dir = DATA_ROOT / sample_id
    write_filtered_10x(stem, selected.obs_names.astype(str).tolist(), sample_dir)

    selected.obs[["cell_class", "is_noise", "pbs-2"]].to_csv(
        sample_dir / "cell_classes.csv"
    )

    summary_rows.append(
        {
            "sample_id": sample_id,
            "n_total_raw": full.n_obs,
            "n_noise": n_noise,
            "n_qc_pass": n_qc,
            "n_pbs2_added": n_pbs2,
            "n_kept": selected.n_obs,
        }
    )
    print(f"  kept {selected.n_obs:,} (qc_pass={n_qc:,}, pbs-2={n_pbs2:,})")

summary = pd.DataFrame(summary_rows)
summary_path = DATA_ROOT / "cell_inclusion_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print("Total kept:", int(summary["n_kept"].sum()), "| PBS-2 added:", int(summary["n_pbs2_added"].sum()))
print("Wrote", summary_path)

# SampleStats still written for provenance; R will skip applying it.
ann = pd.read_csv(ANNOTATIONS).set_index("sample-name")
stats_rows = []
for sample_id in SAMPLE_MAP:
    row = ann.loc[sample_id]
    stats_rows.append(
        {
            "SampleName": sample_id,
            "Mito": float(row["mito-upper"]),
            "GeneLower": int(row["genes-lower"]),
            "GeneUpper": int(row["genes-upper"]),
            "LibSize": int(row["library-size-upper"]),
        }
    )
pd.DataFrame(stats_rows).to_csv(DATA_ROOT / "SampleStats.txt", sep="\t", index=False)

### Optional: NCBI gene-info file (author symbol remap)

Same as the null notebook. If absent, the R script keeps Cell Ranger symbols as `Official`.

In [ ]:
gene_info = DATA_ROOT / "180808_Homo_sapiens.gene_info.gz"
if gene_info.exists():
    print("Found", gene_info)
else:
    print("Optional gene-info not present; R will skip alias remap.")

## 2. Run Seurat + edgeR pipeline (skip cell QC)

Same `NormBRCA1_faithful.R` as the null notebook, with third argument `TRUE` so SampleStats cell filters are not re-applied.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

def find_rscript() -> Path | None:
    """Locate an absolute Rscript path usable by this kernel."""
    candidates: list[Path] = []

    # 1) Same bin/ as this kernel's python (most reliable on SageMaker)
    py_bin = Path(sys.executable).resolve().parent
    candidates.append(py_bin / "Rscript")
    candidates.append(py_bin.parent / "bin" / "Rscript")

    conda_prefix = os.environ.get("CONDA_PREFIX")
    if conda_prefix:
        candidates.append(Path(conda_prefix) / "bin" / "Rscript")

    which = shutil.which("Rscript")
    if which:
        candidates.append(Path(which).resolve())

    home = Path.home()
    for pattern in (
        "anaconda3/envs/*/bin/Rscript",
        "miniconda3/envs/*/bin/Rscript",
        "SageMaker/.conda/envs/*/bin/Rscript",
        ".conda/envs/*/bin/Rscript",
    ):
        candidates.extend(home.glob(pattern))

    candidates.extend(
        [
            Path("/opt/conda/bin/Rscript"),
            Path("/opt/conda/envs/signals-in-the-noise/bin/Rscript"),
            Path("/usr/bin/Rscript"),
            Path("/usr/local/bin/Rscript"),
        ]
    )

    seen: set[str] = set()
    for c in candidates:
        try:
            c = Path(c).expanduser().resolve()
        except OSError:
            continue
        key = str(c)
        if key in seen:
            continue
        seen.add(key)
        # Absolute existing file only — never accept bare "Rscript".
        # Do NOT require os.X_OK (SageMaker/NFS often strips the exec bit in access()).
        if c.is_absolute() and c.is_file():
            return c
    return None


def install_r_into_kernel_env() -> Path | None:
    """Install r-base into the same conda prefix as this notebook kernel."""
    prefix = Path(sys.executable).resolve().parent.parent
    print(f"Installing r-base into kernel env: {prefix}")
    conda = shutil.which("conda")
    if conda is None:
        for cand in (
            Path(sys.executable).resolve().parent / "conda",
            Path("/opt/conda/bin/conda"),
            Path.home() / "anaconda3" / "bin" / "conda",
            Path.home() / "miniconda3" / "bin" / "conda",
        ):
            if cand.is_file():
                conda = str(cand)
                break
    if conda is None:
        print("conda not found; cannot auto-install r-base")
        return None
    cmd = [
        conda,
        "install",
        "-y",
        "-c",
        "conda-forge",
        "-p",
        str(prefix),
        "r-base",
    ]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, check=False)
    print("conda exit:", completed.returncode)
    return find_rscript()


print("sys.executable:", sys.executable)
print("kernel env prefix:", Path(sys.executable).resolve().parent.parent)
print("CONDA_PREFIX:", os.environ.get("CONDA_PREFIX"))
print("PATH head:", os.environ.get("PATH", "")[:200])

rscript = find_rscript()
if rscript is None:
    print("Rscript not found — attempting conda install into THIS kernel's env...")
    rscript = install_r_into_kernel_env()

print("Rscript:", rscript)
if rscript is None or not Path(rscript).is_absolute() or not Path(rscript).is_file():
    prefix = Path(sys.executable).resolve().parent.parent
    raise RuntimeError(
        "Could not resolve an absolute Rscript path for this kernel.\n\n"
        "In a Terminal on this instance, run:\n"
        f"  conda install -y -c conda-forge -p {prefix} r-base\n"
        f"  ls {prefix}/bin/Rscript\n"
        "Then: Kernel → Restart Kernel, re-run from the top.\n\n"
        "Or set manually in a cell before the run step:\n"
        f"  rscript = Path('{prefix}/bin/Rscript')"
    )

# Never call a bare command name — only an absolute file path.
rscript = Path(rscript).resolve()
assert rscript.is_absolute() and rscript.is_file(), rscript
print("Using absolute Rscript:", rscript)
subprocess.run([str(rscript), "--version"], check=True)
print("OK")


In [ ]:
import subprocess
from pathlib import Path

assert R_SCRIPT.exists(), f"Missing R script: {R_SCRIPT}"

if "rscript" not in dir() or rscript is None:
    raise RuntimeError("Set rscript to an absolute Path first (run the previous cell).")

rscript = Path(rscript).expanduser().resolve()
if not rscript.is_absolute() or not rscript.is_file():
    raise FileNotFoundError(
        f"rscript must be an absolute existing file, got: {rscript!r}. "
        "Run the check-rscript cell, or set "
        "rscript = Path('/full/path/to/bin/Rscript')"
    )
if rscript.name == "Rscript" and str(rscript) == "Rscript":
    raise FileNotFoundError("Refusing bare 'Rscript'; set an absolute path.")

cmd = [str(rscript), str(R_SCRIPT), str(DATA_ROOT), str(OUT_DIR), "TRUE"]
print("Running:\n ", " ".join(cmd))
completed = subprocess.run(cmd, check=False)
print("exit code:", completed.returncode)
if completed.returncode != 0:
    raise RuntimeError(
        f"NormBRCA1_faithful.R failed with exit code {completed.returncode}. "
        "Check the R console output above."
    )


## 3. Outputs

Compare against the null run in `brca1_preneoplastic_seurat/output/`.

| File | Content |
|---|---|
| `Fig4A.pdf` / `Fig4B.pdf` / `Fig4C.pdf` | t-SNE panels with PBS-2 included |
| `FigS1A-*.pdf` / `FigS1C.pdf` | Proportions + pseudo-bulk heatmap |
| `SeuratObject_NormB1Total.rds` | Integrated object |
| `../Data/cell_inclusion_summary.csv` | QC-pass vs PBS-2 counts per sample |
| `../Data/<sample>/cell_classes.csv` | Per-barcode class labels |

In [ ]:
for p in sorted(OUT_DIR.glob("*")):
    print(f"{p.name:40s} {p.stat().st_size/1e6:8.2f} MB")

counts_path = OUT_DIR / "cluster_by_condition_counts.csv"
if counts_path.exists():
    display(pd.read_csv(counts_path, index_col=0))

In [ ]:
pdfs = ["Fig4A.pdf", "Fig4B.pdf", "Fig4C.pdf", "FigS1A-right.pdf", "FigS1C.pdf"]
for name in pdfs:
    pdf = OUT_DIR / name
    if not pdf.exists():
        print("missing", name)
        continue
    png = OUT_DIR / (pdf.stem + ".png")
    if not png.exists():
        converter = shutil.which("pdftoppm") or shutil.which("magick")
        if converter and "pdftoppm" in Path(converter).name:
            subprocess.run(
                [converter, "-png", "-singlefile", "-r", "120", str(pdf), str(OUT_DIR / pdf.stem)],
                check=False,
            )
        elif converter:
            subprocess.run([converter, "convert", "-density", "120", str(pdf), str(png)], check=False)
    if png.exists():
        print(name)
        display(Image(filename=str(png)))
    else:
        print(f"{name} ready at {pdf}")

## 4. Compare to null (paper QC only)

After both notebooks have finished, contrast:

1. **Cell totals** — `cell_inclusion_summary.csv` vs null run cell counts.
2. **t-SNE / clusters** — Fig 4A–C side by side (new clusters? PBS-2 enrichment?).
3. **Composition test** — `cluster_by_condition_counts.csv` Normal vs BRCA1.
4. **Appendix S1C** — whether stromal/immune markers still lack WT vs BRCA1 contrast.

Optional follow-up: join `cell_classes.csv` barcodes onto the Seurat object metadata to color t-SNE by `qc_pass` vs `pbs-2`.